# Day 7 Lab 02: Spark SQL Exploration

        **Layer:** Spark fundamentals  
        **Python reference:** `Lab_Files/labs/lab_02_spark_sql_exploration.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Register raw order events as a temp view.
- Run SQL aggregations in Spark.
- Write and inspect status-level event counts.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [1]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Accer\Documents\GitHub\Medallion-pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Accer\Documents\GitHub\Medallion-pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [2]:

import importlib
import day7_common
day7_common = importlib.reload(day7_common)

from day7_common import OUTPUT_DIR, ORDER_SOURCE_FILES, ensure_output_dirs, read_order_events, require_source_data, spark_session, write_csv_dir

## 2. Start Spark

In [3]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook02SparkSQL")

Py4JError: An error occurred while calling None.org.apache.spark.sql.SparkSession. Trace:
py4j.Py4JException: Constructor org.apache.spark.sql.SparkSession([class org.apache.spark.SparkContext, class java.util.HashMap]) does not exist
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:180)
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:197)
	at py4j.Gateway.invoke(Gateway.java:237)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:108)
	at java.base/java.lang.Thread.run(Thread.java:842)



In [4]:
import pyspark
print(pyspark.__version__)

3.4.0


## 3. Read Batch 1 Raw Events

In [4]:
orders = read_order_events(spark, [ORDER_SOURCE_FILES[0]])
orders.printSchema()
orders.show(10, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- delivery_promise: string (nullable = true)

+--------+--------+-----------+----------+---------+-------+--------+-------+--------------------+-----------+----------------+
|event_id|order_id|customer_id|product_id|status   |amount |currency|channel|event_time          |coupon_code|delivery_promise|
+--------+--------+-----------+----------+---------+-------+--------+-------+--------------------+-----------+----------------+
|evt-1001|1001    |501        |P-LAP-01  |NEW      |1299.99|USD     |web    |2026-05-29T02:00:01Z|null       |null            |
|evt-1002|1002    |502     

## 4. Register a Temp View

In [5]:
orders.createOrReplaceTempView("raw_orders")

## 5. Run Spark SQL

In [6]:
status_counts = spark.sql(
    '''
    SELECT
      status,
      COUNT(*) AS event_count,
      ROUND(SUM(amount), 2) AS raw_amount_total
    FROM raw_orders
    GROUP BY status
    ORDER BY event_count DESC, status
    '''
)

In [7]:
status_counts.show()

+---------+-----------+----------------+
|   status|event_count|raw_amount_total|
+---------+-----------+----------------+
|      NEW|          5|         3037.49|
|CANCELLED|          1|           450.0|
+---------+-----------+----------------+



## 6. Inspect and Write Output

In [8]:
status_counts.show(truncate=False)
write_csv_dir(status_counts, OUTPUT_DIR / "notebook_02_status_counts")
print(f"Raw events analyzed from batch 1: {orders.count()}")

+---------+-----------+----------------+
|status   |event_count|raw_amount_total|
+---------+-----------+----------------+
|NEW      |5          |3037.49         |
|CANCELLED|1          |450.0           |
+---------+-----------+----------------+

Raw events analyzed from batch 1: 6


## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [9]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()